In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
import joblib

In [2]:
raw_df = pd.read_csv('weatherAUS.csv')
raw_df = raw_df.dropna(subset=["RainTomorrow"])

In [3]:
raw_df.head()

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


In [4]:
#Визначення колонок

target_col = "RainTomorrow"
categorical_col = ['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainToday']
numeric_col = ['MinTemp','MaxTemp','Rainfall','Evaporation','Sunshine',
    'WindGustSpeed','WindSpeed9am','WindSpeed3pm',
    'Humidity9am','Humidity3pm',
    'Pressure9am','Pressure3pm',
    'Cloud9am','Cloud3pm',
    'Temp9am','Temp3pm'
]

input_cols = categorical_col + numeric_col

In [5]:
X = raw_df[input_cols]
y = (raw_df[target_col] == "Yes").astype(int)

In [6]:
#preprocess

imputer = SimpleImputer()
scaler = MinMaxScaler()
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

X[numeric_col]= imputer.fit_transform(X[numeric_col])

X_scaled = scaler.fit_transform(X[numeric_col])
X_scaled = pd.DataFrame(X_scaled, columns=numeric_col)

# Кодування категоріальних
X_encoded = encoder.fit_transform(X[categorical_col])
encoded_cols = encoder.get_feature_names_out(categorical_col)
X_encoded = pd.DataFrame(X_encoded, columns=encoded_cols)

# Об’єднання
X_ready = pd.concat([X_scaled, X_encoded], axis=1)

/tmp/ipython-input-3206237501.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[numeric_col]= imputer.fit_transform(X[numeric_col])


In [7]:
# 4. Тренування моделей

X_train, X_test, y_train, y_test = train_test_split(X_ready, y, test_size=0.2, random_state=42)

# --- Logistic Regression
log_model = LogisticRegression(max_iter=500, solver='liblinear')
log_model.fit(X_train, y_train)
y_pred_log = log_model.predict(X_test)
y_prob_log = log_model.predict_proba(X_test)[:, 1]

# --- Random Forest
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

In [9]:
# Порівняння результатів

print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_log))
print("ROC AUC:", roc_auc_score(y_test, y_prob_log))
print(classification_report(y_test, y_pred_log))

print("Random Forest")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("ROC AUC:", roc_auc_score(y_test, y_prob_rf))
print(classification_report(y_test, y_pred_rf))


# Збереження кращої моделі

# Якщо Random Forest має вищий ROC AUC:
if roc_auc_score(y_test, y_prob_rf) > roc_auc_score(y_test, y_prob_log):
    best_model = rf_model
    model_name = "aussie_rain_rf.joblib"
    print("\n✅ Random Forest обрано як кращу модель.")
else:
    best_model = log_model
    model_name = "aussie_rain_log.joblib"
    print("\n✅ Logistic Regression залишилась кращою.")

joblib.dump({
    'model': best_model,
    'imputer': imputer,
    'scaler': scaler,
    'encoder': encoder,
    'input_cols': input_cols,
    'numeric_cols': numeric_col,
    'categorical_cols': categorical_col
}, model_name)
print(f"зберігаємо {model_name}")

Logistic Regression
Accuracy: 0.846337775589859
ROC AUC: 0.8681573339868144
              precision    recall  f1-score   support

           0       0.87      0.94      0.90     22098
           1       0.72      0.51      0.60      6341

    accuracy                           0.85     28439
   macro avg       0.79      0.73      0.75     28439
weighted avg       0.84      0.85      0.84     28439

Random Forest
Accuracy: 0.852561623123176
ROC AUC: 0.8833265471728644
              precision    recall  f1-score   support

           0       0.87      0.95      0.91     22098
           1       0.76      0.50      0.60      6341

    accuracy                           0.85     28439
   macro avg       0.81      0.73      0.76     28439
weighted avg       0.84      0.85      0.84     28439


✅ Random Forest обрано як кращу модель.
зберігаємо aussie_rain_rf.joblib
